# Testing & Comparison
### CNN-Only | GRU-Only | CNN+GRU | TCN

Loads the best checkpoint for each model, evaluates on the held-out test set (chb22–24),
and produces a full comparison with all metrics, ROC curves, and per-patient results.

**Prerequisites:** Run notebooks 05 and 06 first to train and save checkpoints.

In [ ]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
import warnings; warnings.filterwarnings('ignore')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, confusion_matrix

from src.models import CNNClassifier, GRUClassifier, CNNGRUClassifier, TCNClassifier, FocalLoss
from src.training.trainer import run_epoch
from src.data.dataset import build_split_dataset, EEGDataset, build_eval_loader

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CKPT_DIR = Path('checkpoints')
THRESHOLD = 0.45

print(f'Device    : {DEVICE}')
print(f'Threshold : {THRESHOLD}')

In [ ]:
with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

_raw  = cfg['data']['raw_dir']
_ch   = cfg['data']['channels']
_wsec = cfg['data']['window_size']
_fs   = cfg['data']['sample_rate']
_ov   = cfg['data']['overlap']
_szt  = cfg['data']['seizure_threshold']
_bpl  = cfg['preprocessing']['bandpass_low']
_bph  = cfg['preprocessing']['bandpass_high']
_ntch = cfg['preprocessing']['notch_freq']
_seed = cfg['training']['seed']
_bs   = cfg['training']['batch_size']

# Save to /tmp — no quota, survives the session
CACHE_DIR = '/tmp/neuroscribe_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

cache_path = os.path.join(CACHE_DIR, 'test_subsampled.npz')
if os.path.exists(cache_path):
    print('Loading test split from /tmp cache...')
    d = np.load(cache_path)
    test_ds = EEGDataset(d['windows'], d['labels'], d['patient_ids'])
else:
    print('Building test split from raw EDF (first run only)...')
    test_ds_full = build_split_dataset(
        raw_dir=_raw, patient_ids=cfg['splits']['test_patients'], split_name='test',
        target_channels=_ch, window_size_sec=_wsec, overlap=_ov,
        seizure_threshold=_szt, bandpass_low=_bpl, bandpass_high=_bph,
        notch_freq=_ntch, sample_rate=_fs,
        processed_dir=None, use_cache=False,
    )
    np.random.seed(_seed)
    s_idx  = np.where(test_ds_full.labels == 1)[0]
    ns_all = np.where(test_ds_full.labels == 0)[0]
    ns_idx = np.random.choice(ns_all, min(len(s_idx) * 5, len(ns_all)), replace=False)
    idx    = np.random.permutation(np.concatenate([s_idx, ns_idx]))
    test_ds = EEGDataset(test_ds_full.windows[idx], test_ds_full.labels[idx], test_ds_full.patient_ids[idx])
    np.savez_compressed(cache_path, windows=test_ds.windows, labels=test_ds.labels, patient_ids=test_ds.patient_ids)
    print(f'  Cached → {cache_path}')

test_loader = build_eval_loader(test_ds, batch_size=_bs * 2)
print(f'Test set: {len(test_ds):,} windows  |  '
      f'{test_ds.n_seizure:,} seizure ({test_ds.seizure_fraction:.2%})')

---
## Load All Models from Checkpoints

In [ ]:
MODELS = {
    'CNN-Only' : CNNClassifier  (n_channels=23, n_samples=1024, dropout=0.5),
    'GRU-Only' : GRUClassifier  (n_channels=23, hidden_size=128, num_layers=2, dropout=0.5),
    'CNN+GRU'  : CNNGRUClassifier(n_channels=23, dropout=0.5),
    'TCN'      : TCNClassifier  (n_channels=23, proj_channels=64, num_filters=128,
                                  kernel_size=3, num_blocks=8, dropout=0.5),
}

print('Loading checkpoints...')
for name, model in MODELS.items():
    ckpt = CKPT_DIR / f'{name.replace("+","_").replace("-","_")}_best.pt'
    if ckpt.exists():
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        model.to(DEVICE)
        params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'  {name:<12}  ✓ loaded  ({params:,} params)')
    else:
        print(f'  {name:<12}  ✗ checkpoint not found: {ckpt}')

---
## Batch Inference Demo

In [ ]:
demo_X, demo_y = next(iter(test_loader))
demo_X, demo_y = demo_X.to(DEVICE), demo_y.to(DEVICE)

print(f'Demo batch : {demo_X.shape}  (batch={demo_X.shape[0]}, C=23, T=1024)')
print(f'Ground truth: {int(demo_y.sum())} seizure / {int((1-demo_y).sum())} non-seizure')
print(f'Threshold   : {THRESHOLD}\n')

demo_results = {}
for name, model in MODELS.items():
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(demo_X)).cpu().numpy()
    preds = (probs >= THRESHOLD).astype(int)
    gt    = demo_y.cpu().numpy().astype(int)
    tp = int(((preds==1)&(gt==1)).sum())
    fp = int(((preds==1)&(gt==0)).sum())
    fn = int(((preds==0)&(gt==1)).sum())
    tn = int(((preds==0)&(gt==0)).sum())
    sens = tp / max(tp+fn, 1); prec = tp / max(tp+fp, 1)
    f1   = 2*sens*prec / max(sens+prec, 1e-8)
    demo_results[name] = dict(probs=probs, tp=tp, fp=fp, fn=fn, tn=tn, sens=sens, prec=prec, f1=f1)
    print(f'  {name:<12}  TP={tp} FP={fp} FN={fn} TN={tn}  Sens={sens:.3f}  Prec={prec:.3f}  F1={f1:.3f}')

In [ ]:
colors = {'CNN-Only':'#e67e22', 'GRU-Only':'#2980b9', 'CNN+GRU':'#27ae60', 'TCN':'#8e44ad'}
gt = demo_y.cpu().numpy()
n  = len(gt)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
for ax, (name, res) in zip(axes, demo_results.items()):
    for i in range(n):
        ax.axvspan(i-.5, i+.5, color='salmon' if gt[i]==1 else 'lightblue', alpha=0.3)
    ax.bar(range(n), res['probs'], color=colors[name], alpha=0.85, width=0.6)
    ax.axhline(THRESHOLD, color='red', linestyle='--', lw=1.2, label=f'τ={THRESHOLD}')
    ax.set_ylim(0, 1.05); ax.set_ylabel('P(seizure)')
    ax.set_title(f'{name}  F1={res["f1"]:.3f}  Sens={res["sens"]:.3f}  Prec={res["prec"]:.3f}',
                 fontweight='bold')
    ax.legend(fontsize=8, loc='upper right'); ax.grid(axis='y', alpha=0.3)

axes[-1].set_xlabel('Window index')
plt.suptitle('Batch Inference Demo — Red bg=seizure  Blue bg=non-seizure', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('Report_Figures/demo_batch_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Full Test Set Evaluation

In [ ]:
def evaluate(name, model, loader, device, threshold=0.45):
    metrics = run_epoch(model, loader, FocalLoss(alpha=0.85, gamma=2.0), None, device, threshold)
    probs, labels = metrics['probs'], metrics['labels']
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()
    total_hours = len(labels) * 4 / 3600
    return {
        'sensitivity': metrics['sensitivity'],
        'precision':   metrics['precision'],
        'specificity': tn / (tn + fp + 1e-9),
        'f1':          metrics['f1'],
        'roc_auc':     roc_auc_score(labels, probs),
        'pr_auc':      average_precision_score(labels, probs),
        'fp_per_hr':   fp / total_hours,
        'probs':       probs,
        'labels':      labels,
    }

print(f'Evaluating all models on test set (τ={THRESHOLD})...')
test_results = {}
for name, model in MODELS.items():
    test_results[name] = evaluate(name, model, test_loader, DEVICE, THRESHOLD)
    r = test_results[name]
    print(f'  {name:<12}  F1={r["f1"]:.4f}  Sens={r["sensitivity"]:.4f}  '
          f'Spec={r["specificity"]:.4f}  AUC={r["roc_auc"]:.4f}  FP/hr={r["fp_per_hr"]:.2f}')

---
## Comparison Results Table

In [ ]:
rows = []
for name, r in test_results.items():
    params = sum(p.numel() for p in MODELS[name].parameters() if p.requires_grad)
    rows.append({'Model': name, 'Params': f'{params/1e3:.0f}K',
                 'Sensitivity': f'{r["sensitivity"]:.4f}',
                 'Precision':   f'{r["precision"]:.4f}',
                 'Specificity': f'{r["specificity"]:.4f}',
                 'F1':          f'{r["f1"]:.4f}',
                 'ROC AUC':     f'{r["roc_auc"]:.4f}',
                 'PR AUC':      f'{r["pr_auc"]:.4f}',
                 'FP/hr':       f'{r["fp_per_hr"]:.2f}'})

df = pd.DataFrame(rows).set_index('Model')
print('\nTest Set Results (chb22–24)\n')
print(df.to_string())
df

---
## Visual Comparison

In [ ]:
model_names = list(test_results.keys())
color_list  = [colors[n] for n in model_names]
x = np.arange(len(model_names))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# F1 / Sensitivity / Specificity grouped bar
bar_w = 0.25
f1s   = [test_results[n]['f1']          for n in model_names]
senss = [test_results[n]['sensitivity'] for n in model_names]
specs = [test_results[n]['specificity'] for n in model_names]
for offset, vals, label, c in zip([-bar_w, 0, bar_w],
                                   [f1s, senss, specs],
                                   ['F1', 'Sensitivity', 'Specificity'],
                                   ['steelblue', 'darkorange', 'seagreen']):
    bars = axes[0].bar(x + offset, vals, bar_w, label=label, color=c, alpha=0.85)
    for bar in bars:
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                     f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=7)
axes[0].set_xticks(x); axes[0].set_xticklabels(model_names)
axes[0].set_ylim(0, 1.15); axes[0].legend(fontsize=9)
axes[0].set_title('F1 / Sensitivity / Specificity', fontweight='bold'); axes[0].grid(axis='y', alpha=0.3)

# ROC AUC bar
aucs = [test_results[n]['roc_auc'] for n in model_names]
bars = axes[1].bar(model_names, aucs, color=color_list, edgecolor='black', alpha=0.85)
for bar, val in zip(bars, aucs):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].set_ylim(0.5, 1.05); axes[1].set_title('ROC AUC', fontweight='bold'); axes[1].grid(axis='y', alpha=0.3)

# ROC curves
axes[2].plot([0,1],[0,1],'k--', alpha=0.4, label='Random')
for name in model_names:
    r = test_results[name]
    fpr, tpr, _ = roc_curve(r['labels'], r['probs'])
    axes[2].plot(fpr, tpr, color=colors[name], lw=2, label=f'{name} ({r["roc_auc"]:.3f})')
axes[2].set_xlabel('FPR'); axes[2].set_ylabel('TPR')
axes[2].set_title('ROC Curves', fontweight='bold'); axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)

plt.suptitle('Model Comparison — Test Set (chb22–24)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('Report_Figures/comparison_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → Report_Figures/comparison_results.png')

---
## Performance Improvement Summary

In [ ]:
tcn = test_results['TCN']
baseline = test_results['CNN+GRU']

print('Performance Improvement: TCN vs CNN+GRU (Ckpt 2 baseline)')
print('='*55)
for metric in ['f1', 'sensitivity', 'specificity', 'roc_auc', 'pr_auc']:
    delta = tcn[metric] - baseline[metric]
    sign  = '+' if delta >= 0 else ''
    better = '✓' if delta > 0 else ('=' if delta == 0 else '✗')
    print(f'  {better} {metric:<12}  CNN+GRU={baseline[metric]:.4f}  '
          f'TCN={tcn[metric]:.4f}  (Δ={sign}{delta:.4f})')

print()
print('Why TCN outperforms:')
print('  1. Full 4s receptive field via 8 dilated blocks (1021 samples)')
print('  2. Parallel computation — no vanishing gradient across long sequences')
print('  3. Residual connections for stable 8-block deep training')
print('  4. Input noise augmentation reduces overfitting to training patients')
print('  5. Cosine warm restart LR avoids local minima')